# SignGuard AI — Traffic Sign Recognition with a Safety Audit

**Part A — the model.** MobileNetV2 transfer learning on GTSRB (43 German traffic sign classes).
Same architecture and training recipe as my MenuDetector project.

**Part B — the safety audit.** This is what the project is actually about. A classifier always
answers, even when it should not. Part B measures *when the model should be trusted* and teaches
it to say **"I don't know"**:

1. **Calibration** — is a 95% confidence really 95% correct? (reliability diagram, ECE, temperature scaling)
2. **Out-of-distribution rejection** — the camera sees a billboard, not a sign. Does the model notice? (max-softmax, energy score, AUROC)
3. **Occlusion robustness** — the sticker-on-a-stop-sign attack. How fast does accuracy collapse, and does confidence collapse with it?

The deliverable is not an accuracy number. It is a decision rule with measured trade-offs.

---
**Runtime:** `Runtime -> Change runtime type -> T4 GPU`

## 0. Setup

In [ ]:
import os, json, random, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU detected. Runtime -> Change runtime type -> T4 GPU')

OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Dataset — GTSRB

The German Traffic Sign Recognition Benchmark: ~39k training images, 12,630 test images,
43 classes. `torchvision` downloads it directly, so there is no Kaggle token to set up.

The official test split was photographed on **different roads on different days** than the
training split. That matters: the test accuracy below is a real generalisation number, not a
reshuffle of the same photos.

In [ ]:
CLASS_NAMES = [
    'Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)',
    'Speed limit (60km/h)', 'Speed limit (70km/h)', 'Speed limit (80km/h)',
    'End of speed limit (80km/h)', 'Speed limit (100km/h)', 'Speed limit (120km/h)',
    'No passing', 'No passing for vehicles over 3.5t',
    'Right-of-way at next intersection', 'Priority road', 'Yield', 'Stop',
    'No vehicles', 'Vehicles over 3.5t prohibited', 'No entry', 'General caution',
    'Dangerous curve to the left', 'Dangerous curve to the right', 'Double curve',
    'Bumpy road', 'Slippery road', 'Road narrows on the right', 'Road work',
    'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
    'Beware of ice/snow', 'Wild animals crossing',
    'End of all speed and passing limits', 'Turn right ahead', 'Turn left ahead',
    'Ahead only', 'Go straight or right', 'Go straight or left', 'Keep right',
    'Keep left', 'Roundabout mandatory', 'End of no passing',
    'End of no passing by vehicles over 3.5t',
]
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES == 43
print(NUM_CLASSES, 'classes')

In [ ]:
IMG_SIZE = 224                      # MobileNetV2 input size
MEAN = [0.485, 0.456, 0.406]        # ImageNet statistics
STD  = [0.229, 0.224, 0.225]

# NOTE: no horizontal flip. Flipping a traffic sign changes its meaning
# ("turn left ahead" becomes "turn right ahead"), so it would teach the model wrong labels.
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# un-normalised, used later for the occlusion attack
raw_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
DATA_ROOT = './data'

train_full_aug  = datasets.GTSRB(DATA_ROOT, split='train', download=True,  transform=train_tf)
train_full_eval = datasets.GTSRB(DATA_ROOT, split='train', download=False, transform=eval_tf)
test_set        = datasets.GTSRB(DATA_ROOT, split='test',  download=True,  transform=eval_tf)
test_raw        = datasets.GTSRB(DATA_ROOT, split='test',  download=False, transform=raw_tf)

# 85 / 15 train-validation split of the training set.
# Two dataset objects share the same files but different transforms, so the
# validation images are never augmented.
n = len(train_full_aug)
perm = np.random.RandomState(SEED).permutation(n)
n_val = int(0.15 * n)
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_set = Subset(train_full_aug,  train_idx)
val_set   = Subset(train_full_eval, val_idx)

print(f'train {len(train_set):>6}  val {len(val_set):>6}  test {len(test_set):>6}')

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(train_set, True)
val_loader   = make_loader(val_set,   False)
test_loader  = make_loader(test_set,  False)
print('batches/epoch:', len(train_loader))

### 1.1 Look at the data before training anything

In [ ]:
def denorm(t):
    x = t.clone()
    for c in range(3):
        x[c] = x[c] * STD[c] + MEAN[c]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, img, y in zip(axes.ravel(), imgs, labels):
    ax.imshow(denorm(img))
    ax.set_title(CLASS_NAMES[y][:22], fontsize=8)
    ax.axis('off')
fig.suptitle('GTSRB training batch (augmented)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Class balance. GTSRB is strongly imbalanced -- some signs are 10x rarer than others.
# Worth knowing now, because rare classes are exactly where confidence will later be worst.
try:
    all_labels = np.array([lbl for _, lbl in train_full_aug._samples])
except AttributeError:
    all_labels = np.array([train_full_aug[i][1] for i in range(len(train_full_aug))])

counts = np.bincount(all_labels, minlength=NUM_CLASSES)
order = np.argsort(counts)

plt.figure(figsize=(13, 4))
plt.bar(range(NUM_CLASSES), counts[order], color='#4C78A8')
plt.xticks(range(NUM_CLASSES), [CLASS_NAMES[i][:18] for i in order],
           rotation=90, fontsize=7)
plt.ylabel('training images')
plt.title(f'Class imbalance: {counts.min()} to {counts.max()} images per class '
          f'({counts.max() / counts.min():.1f}x)')
plt.tight_layout(); plt.show()

## 2. Model — MobileNetV2 transfer learning

Identical recipe to MenuDetector: load ImageNet weights, freeze the convolutional backbone,
replace the final linear layer with one that has 43 outputs.

The one addition is **phase 2**: after the head has settled, the last few backbone blocks are
unfrozen at a 10x smaller learning rate. Traffic signs look nothing like ImageNet photos, so the
frozen features alone leave several points of accuracy on the table.

In [ ]:
def build_model(num_classes=NUM_CLASSES):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.features.parameters():
        p.requires_grad = False
    m.classifier[1] = nn.Linear(m.last_channel, num_classes)
    return m

model = build_model().to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'trainable {trainable:,} / {total:,} parameters ({100*trainable/total:.1f}%)')

In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    loss_sum, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * y.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            seen     += y.size(0)
    return loss_sum / seen, correct / seen


def fit(model, optimizer, epochs, history, tag, ckpt='best.pth'):
    best = history['best_val'] if 'best_val' in history else 0.0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
        flag = ''
        if va_acc > best:
            best = va_acc
            torch.save(model.state_dict(), ckpt)
            flag = '  <- saved'
        print(f'[{tag} {ep}/{epochs}] train {tr_loss:.4f}/{tr_acc:.4f}  '
              f'val {va_loss:.4f}/{va_acc:.4f}  {time.time()-t0:.0f}s{flag}')
    history['best_val'] = best
    return history

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

In [ ]:
# Phase 1 -- head only
EPOCHS_HEAD = 4
opt = optim.Adam(model.classifier.parameters(), lr=1e-3)
history = fit(model, opt, EPOCHS_HEAD, history, 'head')

In [ ]:
# Phase 2 -- unfreeze the last backbone blocks and fine-tune gently
EPOCHS_FT = 4
for p in model.features[-4:].parameters():
    p.requires_grad = True

opt = optim.Adam([
    {'params': model.features[-4:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(),    'lr': 3e-4},
])
history = fit(model, opt, EPOCHS_FT, history, 'finetune')

model.load_state_dict(torch.load('best.pth', map_location=DEVICE))
print('best validation accuracy:', round(history['best_val'], 4))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
e = range(1, len(history['train_loss']) + 1)
a1.plot(e, history['train_loss'], label='train'); a1.plot(e, history['val_loss'], label='val')
a1.axvline(EPOCHS_HEAD + 0.5, ls='--', c='grey', lw=1)
a1.set_title('Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.plot(e, history['train_acc'], label='train'); a2.plot(e, history['val_acc'], label='val')
a2.axvline(EPOCHS_HEAD + 0.5, ls='--', c='grey', lw=1)
a2.set_title('Accuracy  (dashed line = fine-tuning starts)')
a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=130); plt.show()

## 3. Test-set evaluation

Logits for the whole test set are cached here. Every experiment in Part B reuses this tensor, so
the audit runs in seconds instead of re-running the model each time.

In [ ]:
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        L.append(model(x.to(DEVICE, non_blocking=True)).cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y)

test_logits, test_labels = collect_logits(model, test_loader)
val_logits,  val_labels  = collect_logits(model, val_loader)

test_acc = (test_logits.argmax(1) == test_labels).float().mean().item()
print(f'TEST ACCURACY: {test_acc*100:.2f}%   ({len(test_labels)} images)')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

pred = test_logits.argmax(1).numpy()
true = test_labels.numpy()
cm = confusion_matrix(true, pred, labels=range(NUM_CLASSES))

plt.figure(figsize=(9, 8))
plt.imshow(np.log1p(cm), cmap='Blues')
plt.colorbar(label='log(1 + count)')
plt.xlabel('predicted'); plt.ylabel('true')
plt.title(f'Confusion matrix — test accuracy {test_acc*100:.2f}%')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=130); plt.show()

# The confusions that actually matter
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:10]
print('\nTop confusions:')
for t, p in pairs:
    if off[t, p] == 0:
        break
    print(f'  {off[t,p]:>4}x   {CLASS_NAMES[t][:32]:<34} -> {CLASS_NAMES[p][:32]}')

In [ ]:
per_class_acc = cm.diagonal() / np.maximum(cm.sum(1), 1)
worst = np.argsort(per_class_acc)[:8]
print('Weakest classes:')
for i in worst:
    print(f'  {per_class_acc[i]*100:5.1f}%   {CLASS_NAMES[i]}   '
          f'(train images: {counts[i]})')

In [ ]:
CKPT = 'signguard_mobilenetv2.pth'
torch.save({
    'state_dict': model.state_dict(),
    'classes':    CLASS_NAMES,
    'img_size':   IMG_SIZE,
    'mean':       MEAN,
    'std':        STD,
    'test_acc':   test_acc,
}, CKPT)
print('saved', CKPT, f'({os.path.getsize(CKPT)/1e6:.1f} MB)')

---
# Part B — the safety audit

Part A produced a number: test accuracy. That number hides three failures that matter far more
than the last two points of accuracy.

| Failure | What it looks like in a car |
|---|---|
| Overconfidence | The model says 97% and is wrong. Nothing downstream can tell. |
| No "I don't know" | A billboard is forced into one of 43 sign classes. |
| Silent collapse | A sticker on a STOP sign flips the label — and confidence stays high. |

Everything below measures those three, then turns the measurements into one decision rule.

## 4. Calibration — is 95% confidence really 95% correct?

A softmax output is not a probability until it has been checked. **Expected Calibration Error
(ECE)** bins predictions by confidence and measures the gap between confidence and accuracy in
each bin. A reliability diagram draws the same thing.

In [ ]:
def ece_and_bins(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    correct = pred.eq(labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i].item(), edges[i + 1].item()
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            rows.append((lo, hi, 0, float('nan'), float('nan')))
            continue
        acc_b  = correct[m].mean().item()
        conf_b = conf[m].mean().item()
        w      = m.float().mean().item()
        ece   += w * abs(acc_b - conf_b)
        rows.append((lo, hi, int(m.sum()), acc_b, conf_b))
    return ece, rows


def reliability_plot(rows, title, ax):
    centers = [(lo + hi) / 2 for lo, hi, *_ in rows]
    accs    = [r[3] for r in rows]
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1, label='perfect')
    ax.bar(centers, accs, width=1 / len(rows) * 0.9,
           color='#4C78A8', edgecolor='white', label='accuracy')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    ax.set_title(title); ax.legend(loc='upper left', fontsize=8); ax.grid(alpha=.25)


probs_raw = F.softmax(test_logits, dim=1)
ece_raw, rows_raw = ece_and_bins(probs_raw, test_labels)
print(f'ECE before calibration: {ece_raw*100:.2f}%')
print(f'mean confidence {probs_raw.max(1).values.mean():.4f}   '
      f'actual accuracy {test_acc:.4f}')

### 4.1 Temperature scaling

One scalar `T` divides the logits before the softmax. It cannot change a single prediction — the
argmax is unchanged, so accuracy is identical — it only rescales how confident the model sounds.

`T` is fitted on the **validation** set by minimising NLL, then applied unchanged to the test set.
Fitting it on the test set would be cheating.

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))   # T = exp(log_t) > 0

    @property
    def T(self):
        return self.log_t.exp()

    def forward(self, logits):
        return logits / self.T


def fit_temperature(logits, labels, max_iter=100):
    scaler = TemperatureScaler()
    nll = nn.CrossEntropyLoss()
    opt = optim.LBFGS([scaler.log_t], lr=0.05, max_iter=max_iter)

    def closure():
        opt.zero_grad()
        loss = nll(scaler(logits), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return scaler.T.detach().item()


TEMPERATURE = fit_temperature(val_logits, val_labels)
print(f'fitted temperature T = {TEMPERATURE:.3f}')
print('  T > 1  ->  the model was overconfident and is being cooled down')
print('  T < 1  ->  the model was underconfident and is being sharpened')

probs_cal = F.softmax(test_logits / TEMPERATURE, dim=1)
ece_cal, rows_cal = ece_and_bins(probs_cal, test_labels)

acc_check = (test_logits.argmax(1) == test_labels).float().mean().item()
print(f'\nECE  {ece_raw*100:.2f}%  ->  {ece_cal*100:.2f}%')
print(f'accuracy unchanged: {acc_check*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
reliability_plot(rows_raw, f'Before  (ECE {ece_raw*100:.2f}%)', axes[0])
reliability_plot(rows_cal, f'After temperature scaling  (ECE {ece_cal*100:.2f}%)', axes[1])
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/calibration.png', dpi=130); plt.show()

## 5. Out-of-distribution rejection — the model has never seen a billboard

The classifier has 43 outputs and no 44th option for "this is not a traffic sign". Shown anything
else, it still picks one of the 43.

Two scores are compared for detecting this:

- **MSP** (maximum softmax probability) — the obvious baseline.
- **Energy** — `T · logsumexp(logits / T)`. It uses the *magnitude* of the logits, not just their
  ratio, and is usually the stronger detector.

Two kinds of out-of-distribution input are tested:

- **far-OOD** — CIFAR-10 photos. Nothing like a sign. The easy case.
- **near-OOD** — real signs with 25% of their surface covered. The hard and realistic case.

In [ ]:
cifar = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=eval_tf)
N_OOD = 2000
ood_subset = Subset(cifar, range(N_OOD))
ood_loader = DataLoader(ood_subset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

ood_logits, _ = collect_logits(model, ood_loader)
print('far-OOD logits:', tuple(ood_logits.shape))

In [ ]:
normalize = transforms.Normalize(MEAN, STD)
PALETTE = np.array([[1., 1., 1.], [0., 0., 0.],
                    [0.90, 0.85, 0.10], [0.10, 0.20, 0.80],
                    [0.85, 0.15, 0.15]], dtype=np.float32)


def apply_stickers(img, frac, rng, n_patches=2, return_cover=False):
    """Cover `frac` of the image area with `n_patches` solid rectangles.

    This is the physical sticker attack in its simplest form: opaque tape or
    vinyl stuck on the sign face.

    The patches may overlap and are clipped at the border, so the area actually
    covered is a little below the nominal `frac` at high values. The true covered
    fraction is measured and reported rather than assumed."""
    if frac <= 0:
        return (img, 0.0) if return_cover else img
    c, h, w = img.shape
    out = img.clone()
    mask = torch.zeros(h, w, dtype=torch.bool)
    area_each = frac * h * w / n_patches
    for _ in range(n_patches):
        ar = rng.uniform(0.4, 2.5)
        ph = max(1, min(int(round(math.sqrt(area_each * ar))), h))
        pw = max(1, min(int(round(math.sqrt(area_each / ar))), w))
        top  = int(rng.integers(0, h - ph + 1))
        left = int(rng.integers(0, w - pw + 1))
        color = torch.from_numpy(PALETTE[rng.integers(len(PALETTE))]).view(3, 1, 1)
        out[:, top:top + ph, left:left + pw] = color
        mask[top:top + ph, left:left + pw] = True
    return (out, mask.float().mean().item()) if return_cover else out


@torch.no_grad()
def occluded_logits(frac, idxs, seed=0, batch=64):
    """Returns (logits, labels, mean_actual_cover)."""
    rng = np.random.default_rng(seed)
    model.eval()
    L, Y, buf, ys, covers = [], [], [], [], []
    for i in idxs:
        img, y = test_raw[int(i)]
        occ_img, cov = apply_stickers(img, frac, rng, return_cover=True)
        covers.append(cov)
        buf.append(normalize(occ_img))
        ys.append(y)
        if len(buf) == batch:
            L.append(model(torch.stack(buf).to(DEVICE)).cpu()); Y += ys; buf, ys = [], []
    if buf:
        L.append(model(torch.stack(buf).to(DEVICE)).cpu()); Y += ys
    return torch.cat(L), torch.tensor(Y), float(np.mean(covers))


N_ATTACK = 2000
attack_idx = np.random.RandomState(SEED).choice(len(test_raw), N_ATTACK, replace=False)
print('attack subset:', N_ATTACK, 'images')

In [ ]:
# What the attack actually looks like
rng_demo = np.random.default_rng(7)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for col, frac in enumerate([0.0, 0.10, 0.20, 0.30, 0.40]):
    for row in range(2):
        img, y = test_raw[int(attack_idx[row])]
        axes[row, col].imshow(apply_stickers(img, frac, rng_demo).permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'{int(frac*100)}% covered', fontsize=10)
fig.suptitle('Simulated sticker attack', fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/occlusion_examples.png', dpi=130); plt.show()

In [ ]:
def msp_score(logits, T=1.0):
    """Maximum softmax probability. Higher = more in-distribution."""
    return F.softmax(logits / T, dim=1).max(1).values

def energy_score(logits, T=1.0):
    """Negative free energy. Higher = more in-distribution."""
    return T * torch.logsumexp(logits / T, dim=1)


from sklearn.metrics import roc_auc_score

def auroc(id_scores, ood_scores):
    y = np.r_[np.ones(len(id_scores)), np.zeros(len(ood_scores))]
    s = np.r_[id_scores.numpy(), ood_scores.numpy()]
    return roc_auc_score(y, s)


near_logits, near_labels, near_cover = occluded_logits(0.25, attack_idx, seed=11)
print(f'near-OOD set: signs with {near_cover*100:.1f}% of their area actually covered\n')

results_ood = {}
for name, fn in [('MSP', msp_score), ('Energy', energy_score)]:
    id_s   = fn(test_logits, TEMPERATURE)
    far_s  = fn(ood_logits,  TEMPERATURE)
    near_s = fn(near_logits, TEMPERATURE)
    results_ood[name] = {
        'far_auroc':  auroc(id_s, far_s),
        'near_auroc': auroc(id_s, near_s),
    }
    print(f'{name:<7} far-OOD AUROC {results_ood[name]["far_auroc"]:.3f}   '
          f'near-OOD AUROC {results_ood[name]["near_auroc"]:.3f}')

print('\n0.5 = coin flip, 1.0 = perfect separation')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, fn) in zip(axes, [('MSP', msp_score), ('Energy', energy_score)]):
    kw = dict(bins=60, alpha=.6, density=True)
    ax.hist(fn(test_logits, TEMPERATURE).numpy(), label='traffic signs (ID)', **kw)
    ax.hist(fn(near_logits, TEMPERATURE).numpy(), label='signs, 25% covered', **kw)
    ax.hist(fn(ood_logits,  TEMPERATURE).numpy(), label='CIFAR-10 (far-OOD)', **kw)
    ax.set_title(f'{name}  —  far {results_ood[name]["far_auroc"]:.3f} / '
                 f'near {results_ood[name]["near_auroc"]:.3f} AUROC')
    ax.set_xlabel(f'{name} score'); ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ood_scores.png', dpi=130); plt.show()

## 6. Occlusion robustness — how fast does it collapse?

The plot below is the core result of the project. Two lines:

- **accuracy** falls as more of the sign is covered — expected.
- **mean confidence on the wrong answers** stays high — not expected, and the actual danger.

The gap between them is the silent-failure zone: the model is wrong and sounds certain.

In [ ]:
FRACS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
occ = {}

for f in FRACS:
    lg, yb, cov = occluded_logits(f, attack_idx, seed=int(f * 1000) + 3)
    p = F.softmax(lg / TEMPERATURE, dim=1)
    conf, pred = p.max(1)
    ok = pred.eq(yb)
    occ[f] = {
        'cover':      cov,                 # measured, not nominal
        'acc':        ok.float().mean().item(),
        'conf_all':   conf.mean().item(),
        'conf_wrong': conf[~ok].mean().item() if (~ok).any() else float('nan'),
        'logits': lg, 'labels': yb,
    }
    print(f'{cov*100:5.1f}% covered   accuracy {occ[f]["acc"]*100:5.1f}%   '
          f'confidence when wrong {occ[f]["conf_wrong"]*100:5.1f}%')

In [ ]:
xs = [occ[f]['cover'] * 100 for f in FRACS]
plt.figure(figsize=(8.5, 4.6))
plt.plot(xs, [occ[f]['acc'] * 100 for f in FRACS], 'o-', lw=2,
         color='#4C78A8', label='accuracy')
plt.plot(xs, [occ[f]['conf_wrong'] * 100 for f in FRACS], 's--', lw=2,
         color='#E45756', label='confidence on wrong predictions')
plt.fill_between(xs, [occ[f]['acc'] * 100 for f in FRACS],
                 [occ[f]['conf_wrong'] * 100 for f in FRACS],
                 color='#E45756', alpha=.10)
plt.xlabel('% of sign surface covered'); plt.ylabel('%')
plt.title('The silent-failure gap')
plt.ylim(0, 102); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/occlusion_curve.png', dpi=130); plt.show()

## 7. The decision rule

Everything above becomes one line of production logic:

```
accept the prediction only if calibrated confidence >= THRESHOLD, otherwise return UNKNOWN
```

The threshold is chosen on the **validation** set to reach a target selective accuracy, then its
real cost and benefit are reported on the test set.

In [ ]:
def coverage_accuracy(logits, labels, T, thresholds):
    p = F.softmax(logits / T, dim=1)
    conf, pred = p.max(1)
    ok = pred.eq(labels)
    cov, sel = [], []
    for t in thresholds:
        m = conf >= t
        cov.append(m.float().mean().item())
        sel.append(ok[m].float().mean().item() if m.any() else float('nan'))
    return np.array(cov), np.array(sel)


TARGET_SELECTIVE_ACC = 0.99
grid = np.linspace(0.0, 0.999, 400)

cov_v, sel_v = coverage_accuracy(val_logits, val_labels, TEMPERATURE, grid)

# Pick the LOWEST threshold (= highest coverage) that reaches the target.
# The selective-accuracy curve is noisy and not perfectly monotonic, so a single
# crossing can be a fluke. Require the target to hold at that threshold and at
# every stricter one. (NaN = no samples left at that threshold, which is not a failure.)
meets = (sel_v >= TARGET_SELECTIVE_ACC) | np.isnan(sel_v)
holds_from_here = np.logical_and.accumulate(meets[::-1])[::-1]
ok_idx = np.where(holds_from_here & ~np.isnan(sel_v))[0]
THRESHOLD = float(grid[ok_idx[0]]) if len(ok_idx) else 0.90

print(f'threshold chosen on validation: {THRESHOLD:.3f} '
      f'(target selective accuracy {TARGET_SELECTIVE_ACC*100:.0f}%)')
print(f'  at that threshold the model answers {cov_v[ok_idx[0]]*100:.1f}% of validation inputs')

cov_t, sel_t = coverage_accuracy(test_logits, test_labels, TEMPERATURE, grid)

plt.figure(figsize=(8, 4.6))
plt.plot(cov_t * 100, sel_t * 100, lw=2, color='#4C78A8')
i = int(np.argmin(np.abs(grid - THRESHOLD)))
plt.scatter([cov_t[i] * 100], [sel_t[i] * 100], zorder=5, s=70, color='#E45756')
plt.annotate(f'  chosen: answer {cov_t[i]*100:.1f}%  /  {sel_t[i]*100:.2f}% correct',
             (cov_t[i] * 100, sel_t[i] * 100), fontsize=9)
plt.xlabel('coverage — % of inputs the model answers')
plt.ylabel('selective accuracy — % correct among answers')
plt.title('What refusing to answer buys you')
plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/coverage_accuracy.png', dpi=130); plt.show()

In [ ]:
def gate(logits, labels=None, T=TEMPERATURE, thr=THRESHOLD):
    p = F.softmax(logits / T, dim=1)
    conf, pred = p.max(1)
    accepted = conf >= thr
    out = {'accept_rate': accepted.float().mean().item()}
    if labels is not None:
        ok = pred.eq(labels)
        out['selective_acc'] = ok[accepted].float().mean().item() if accepted.any() else float('nan')
        out['silent_fail']   = (accepted & ~ok).float().mean().item()
    return out

clean   = gate(test_logits, test_labels)
occ20   = gate(occ[0.20]['logits'], occ[0.20]['labels'])
farood  = gate(ood_logits)

print('CLEAN SIGNS')
print(f'  answered            {clean["accept_rate"]*100:5.1f}%')
print(f'  accuracy on answers {clean["selective_acc"]*100:5.2f}%   (was {test_acc*100:.2f}% with no gate)')
print(f'  silent failures     {clean["silent_fail"]*100:5.2f}%')
print('\n20% COVERED SIGNS (sticker attack)')
print(f'  answered            {occ20["accept_rate"]*100:5.1f}%')
print(f'  silent failures     {occ20["silent_fail"]*100:5.2f}%   (confident AND wrong)')
print('\nNOT A SIGN AT ALL (CIFAR-10)')
print(f'  wrongly answered    {farood["accept_rate"]*100:5.1f}%   (should be 0%)')

## 8. Results table — paste into the README

In [ ]:
no_gate_occ20 = occ[0.20]
p20 = F.softmax(no_gate_occ20['logits'] / TEMPERATURE, dim=1)
wrong20 = (p20.argmax(1) != no_gate_occ20['labels']).float().mean().item()

table = f"""
| Metric | No gate | Confidence gate (T={TEMPERATURE:.2f}, thr={THRESHOLD:.2f}) |
|---|---|---|
| Inputs answered | 100% | {clean['accept_rate']*100:.1f}% |
| Accuracy on answers | {test_acc*100:.2f}% | **{clean['selective_acc']*100:.2f}%** |
| Calibration error (ECE) | {ece_raw*100:.2f}% | **{ece_cal*100:.2f}%** |
| Silent failure, clean signs | {(1-test_acc)*100:.2f}% | **{clean['silent_fail']*100:.2f}%** |
| Silent failure, 20% covered | {wrong20*100:.1f}% | **{occ20['silent_fail']*100:.1f}%** |
| Non-signs wrongly classified | 100% | **{farood['accept_rate']*100:.1f}%** |

OOD detection AUROC — far (CIFAR-10) / near (25% covered signs):
MSP {results_ood['MSP']['far_auroc']:.3f} / {results_ood['MSP']['near_auroc']:.3f} ·
Energy {results_ood['Energy']['far_auroc']:.3f} / {results_ood['Energy']['near_auroc']:.3f}
"""
print(table)
with open(f'{OUT_DIR}/results_table.md', 'w') as f:
    f.write(table)

In [ ]:
# Final artefact: weights + the two numbers that make them safe to deploy.
FINAL = 'signguard_calibrated.pth'
torch.save({
    'state_dict':   model.state_dict(),
    'classes':      CLASS_NAMES,
    'img_size':     IMG_SIZE,
    'mean':         MEAN,
    'std':          STD,
    'temperature':  TEMPERATURE,
    'threshold':    THRESHOLD,
    'test_acc':     test_acc,
    'selective_acc': clean['selective_acc'],
    'coverage':     clean['accept_rate'],
}, FINAL)

audit = {
    'test_accuracy':      round(test_acc, 4),
    'temperature':        round(TEMPERATURE, 4),
    'threshold':          round(THRESHOLD, 4),
    'ece_before':         round(ece_raw, 4),
    'ece_after':          round(ece_cal, 4),
    'coverage':           round(clean['accept_rate'], 4),
    'selective_accuracy': round(clean['selective_acc'], 4),
    'silent_fail_clean':  round(clean['silent_fail'], 4),
    'silent_fail_occ20':  round(occ20['silent_fail'], 4),
    'ood_accept_rate':    round(farood['accept_rate'], 4),
    'auroc': {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in results_ood.items()},
    'occlusion_curve': {str(f): round(occ[f]['acc'], 4) for f in FRACS},
}
with open(f'{OUT_DIR}/audit.json', 'w') as f:
    json.dump(audit, f, indent=2)

print(json.dumps(audit, indent=2))
print('\nDownload these from the Colab file browser:')
print(f'  {FINAL}')
print(f'  {OUT_DIR}/  (audit.json + all figures)')

---
## What this project claims

Not *"94% accurate"*. The claim is:

> On clean signs the gated model answers **X%** of the time and is **Y%** correct when it does.
> Shown something that is not a sign, it stays silent **Z%** of the time. With a quarter of the
> sign taped over, its silent-failure rate drops from **A%** to **B%**.

Every one of those numbers was measured, and the threshold that produces them was chosen on
validation data, never on the test set.

**Honest limitations**

- Simulated stickers are opaque rectangles. Real adversarial patches are optimised against the
  model and are far stronger.
- GTSRB is German signage photographed in daylight. Night, rain and other countries' signs are
  untested — and are exactly the near-OOD cases the gate exists for.
- The gate reduces silent failures; it does not eliminate them. A system built on this still
  needs a second sensor.